# Text Retrieval: Document Vectors, Similarity, and Retrieval Effectiveness

**Companion to slides 20–26** of `Info_retrieval_01.pptx`. This follows the term-weighting notebook. The equations and numerical examples below come from the lecture; executable Python cells verify them. Additional illustrative exercises are labeled separately.

## 1. Queries and documents as vectors (slide 20)

For a collection containing $n$ distinct terms, represent a document and a query as:

$$\mathbf d=(w_1,w_2,\ldots,w_n), \qquad \mathbf q=(q_1,q_2,\ldots,q_n)$$

Here $w_i$ is the weight of term $i$ in the document. The lecture uses the **term-frequency weight** for $q_i$ when term $i$ occurs in the query, and $q_i=0$ otherwise.

The term order must be the same in both vectors.

### Worked example: why we need both document and query vectors

The previous term-weighting lesson explained how we assign term weights to **documents**. The next step is to answer a user's **query**: *Which documents best match the search terms?*

Suppose our collection contains:

- **D1:** "data mining techniques"
- **D2:** "machine learning techniques"
- **D3:** "data mining applications"

Use a common vocabulary in this fixed order:

$$V=(\text{data},\ \text{mining},\ \text{techniques},\ \text{machine},\ \text{learning},\ \text{applications})$$

For illustration, suppose the document weights are:

| Document | data | mining | techniques | machine | learning | applications |
|:--|--:|--:|--:|--:|--:|--:|
| D1 | 0.4 | 0.4 | 0.4 | 0 | 0 | 0 |
| D2 | 0 | 0 | 0.4 | 0.8 | 0.8 | 0 |
| D3 | 0.4 | 0.4 | 0 | 0 | 0 | 0.8 |

**Important:** These values are **illustrative weights**, not TF-IDF values calculated from this three-document collection. Each row is a document vector. For example,

$$\mathbf d_1=(0.4,\ 0.4,\ 0.4,\ 0,\ 0,\ 0)$$

Now suppose the user searches for **"data mining"**. We represent the query in the **same vocabulary and term order**. In this example, we use *binary query weights*: 1 when a term occurs in the query, otherwise 0.

$$\mathbf q=(1,\ 1,\ 0,\ 0,\ 0,\ 0)$$

Thus, the query and documents have the **same six dimensions**, allowing direct comparison. Your lecture slide uses **TF weights** for query terms; binary weights are a simpler demonstration. For a query containing each term once, normalized TF (augmented TF) also gives weight 1 to each present term.

The dot-product similarity of the query with a document is

$$\operatorname{sim}(\mathbf q,\mathbf d_j)=\sum_{i=1}^{6}q_iw_{ji}$$

For example,

$$\operatorname{sim}(\mathbf q,\mathbf d_1)=1(0.4)+1(0.4)+0(0.4)+0(0)+0(0)+0(0)=0.8$$

The other results are:

| Document | Dot-product similarity | Ranking |
|:--|--:|:--|
| D1 | 0.8 | Tied 1st |
| D3 | 0.8 | Tied 1st |
| D2 | 0.0 | 3rd |

**Takeaway:** Term weighting gives us document vectors; the query vector lets us compare a search request with each document and rank the results.

In [ ]:
# Worked example: shared vocabulary, document vectors, and query vector
vocabulary_example = ["data", "mining", "techniques", "machine", "learning", "applications"]

document_vectors_example = {
    "D1": [0.4, 0.4, 0.4, 0.0, 0.0, 0.0],
    "D2": [0.0, 0.0, 0.4, 0.8, 0.8, 0.0],
    "D3": [0.4, 0.4, 0.0, 0.0, 0.0, 0.8],
}

query_text_example = "data mining"
query_terms_example = set(query_text_example.split())
query_vector_example = [1 if term in query_terms_example else 0
                        for term in vocabulary_example]

print("Vocabulary:", vocabulary_example)
print("Query text:", query_text_example)
print("Query vector:", query_vector_example)
print()
print("Document   Dot product")
for doc_name, document_vector in document_vectors_example.items():
    score = sum(q_i * d_i for q_i, d_i in zip(query_vector_example, document_vector))
    print(f"{doc_name:8s}   {score:.1f}")

# Expected ranking: D1 and D3 tie at 0.8, ahead of D2 at 0.0.


In [ ]:
vocabulary = ["data", "mining", "web", "world"]
document_weights = {"data": 0.2, "mining": 0.0, "web": 0.3, "world": 1.0}
query_weights = {"data": 0.75, "mining": 0.75, "web": 0.0, "world": 1.0}
d = [document_weights.get(term, 0.0) for term in vocabulary]
q = [query_weights.get(term, 0.0) for term in vocabulary]
print("Terms:   ", vocabulary)
print("Document:", d)
print("Query:   ", q)

## 2. Dot-product similarity (slides 21–22)

$$\operatorname{sim}(\mathbf q,\mathbf d)=\mathbf q\cdot\mathbf d=\sum_{i=1}^{n}q_iw_i$$

The lecture's example uses:

$$\mathbf d=(0.2,0,0.3,1),\qquad \mathbf q=(0.75,0.75,0,1)$$

$$\operatorname{sim}(\mathbf q,\mathbf d)=(0.75)(0.2)+(0.75)(0)+(0)(0.3)+(1)(1)=\boxed{1.15}$$

**Observations from slide 22:** Shared terms contribute to the score, higher weights contribute more, and longer documents may be favored. Dot-product similarity has no fixed upper bound.

In [ ]:
def dot_product(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have the same length")
    return sum(x * y for x, y in zip(a, b))

contributions = [x * y for x, y in zip(q, d)]
print("Term contributions:")
for term, value in zip(vocabulary, contributions):
    print(f"  {term:8s} {value:.2f}")
print(f"Dot product = {dot_product(q, d):.2f}")

## 3. Cosine similarity (slides 23–24)

Cosine similarity normalizes the dot product by the lengths of **both** vectors:

$$\operatorname{sim}(\mathbf q,\mathbf d)=\cos\theta=\frac{\mathbf q\cdot\mathbf d}{\|\mathbf q\|\,\|\mathbf d\|}$$

where the Euclidean norm is:

$$\|\mathbf d\|=\sqrt{\sum_{i=1}^{n}w_i^2},\qquad \|\mathbf q\|=\sqrt{\sum_{i=1}^{n}q_i^2}$$

For the lecture's vectors:

$$\|\mathbf d\|=\sqrt{0.2^2+0^2+0.3^2+1^2}=\sqrt{1.13}\approx1.063$$

$$\|\mathbf q\|=\sqrt{0.75^2+0.75^2+0^2+1^2}=\sqrt{2.125}\approx1.458$$

$$\cos\theta=\frac{1.15}{(1.063)(1.458)}\approx\boxed{0.742}$$

For nonnegative term weights, cosine similarity lies between 0 and 1. Unlike raw dot product, it accounts for vector length.

In [ ]:
import math

def norm(a):
    return math.sqrt(sum(x * x for x in a))

def cosine_similarity(a, b):
    denominator = norm(a) * norm(b)
    return dot_product(a, b) / denominator if denominator else 0.0

print(f"||d|| = {norm(d):.3f}")
print(f"||q|| = {norm(q):.3f}")
print(f"Cosine similarity = {cosine_similarity(q, d):.3f}")

### Additional illustration: effect of document length

The following **new example** demonstrates why cosine normalization can be useful. Compare $\mathbf d_1=(1,1)$ and $\mathbf d_2=(10,10)$ against $\mathbf q=(1,1)$. They point in the same direction, so their cosine similarities are equal even though their dot products differ.

In [ ]:
query = [1, 1]
for name, doc in [("d1", [1, 1]), ("d2", [10, 10])]:
    print(f"{name}: dot={dot_product(query, doc):.1f}, cosine={cosine_similarity(query, doc):.3f}")

## 4. Retrieval effectiveness: recall and precision (slides 25–26)

The lecture defines a relevant document as one that the user judges useful for the query. Relevance is subjective.

$$\operatorname{Recall}=\frac{\text{number of retrieved relevant documents}}{\text{total number of relevant documents}}$$

$$\operatorname{Precision}=\frac{\text{number of retrieved relevant documents}}{\text{total number of retrieved documents}}$$

**Slide 26 example:** There are 20 relevant documents in the collection. The system retrieves 10 documents, of which 6 are relevant.

$$\operatorname{Recall}=\frac{6}{20}=\boxed{0.30}$$

$$\operatorname{Precision}=\frac{6}{10}=\boxed{0.60}$$

The ideal is recall = 1 and precision = 1, but this is difficult to achieve in practice. The lecture notes that increasing recall often reduces precision.

In [ ]:
def recall_precision(retrieved_relevant, total_relevant, total_retrieved):
    recall = retrieved_relevant / total_relevant if total_relevant else 0.0
    precision = retrieved_relevant / total_retrieved if total_retrieved else 0.0
    return recall, precision

recall, precision = recall_precision(6, 20, 10)
print(f"Recall:    {recall:.2f} ({recall:.0%})")
print(f"Precision: {precision:.2f} ({precision:.0%})")

### Additional illustration: precision–recall as more results are retrieved

This **new example** uses an illustrative ranked result list. `True` means a retrieved document is relevant. We assume there are 6 relevant documents in the full collection.

In [ ]:
ranked_results = [True, True, False, True, False, False, True, False, True, False]
total_relevant = 6
found = 0
print("Top-k   Recall   Precision")
for k, relevant in enumerate(ranked_results, start=1):
    found += int(relevant)
    r, p = recall_precision(found, total_relevant, k)
    print(f"{k:>5}   {r:>6.2f}   {p:>9.2f}")

## 5. Practice questions

1. For $\mathbf q=(1,0,1)$ and $\mathbf d=(2,1,2)$, calculate dot-product and cosine similarity by hand, then check using the Python functions.
2. Why does the dot product change when every document weight is multiplied by 10, while cosine similarity does not?
3. A system retrieves 12 documents, 9 of which are relevant. If the collection contains 30 relevant documents, calculate recall and precision.

**Note:** The notebook retains the lecture's examples and introduces two clearly marked supplementary demonstrations.